In [4]:
import numpy as np
import pandas as pd
from pathlib import Path
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings("ignore")

# =========================
# CONFIG
# =========================
ROOT = Path("/home/mohamed/SDD/hackathon/sia-predicting-short-form-video-popularity")
DATA_DIR = ROOT / "Data"
TRAIN_MERGED = DATA_DIR / "merged_train.csv"
TEST_MERGED  = DATA_DIR / "merged_test.csv"
Y_PATH = ROOT / "y_train.csv"

N_SPLITS = 5
SEED = 42

# =========================
# Utils
# =========================
def read_csv_robust(path: Path) -> pd.DataFrame:
    """Read CSV with auto separator detection + python engine fallback."""
    try:
        return pd.read_csv(path, sep=None, engine="python")
    except Exception:
        try:
            return pd.read_csv(path, sep=";", engine="python")
        except Exception:
            return pd.read_csv(path, sep=",", engine="python")

def normalize_id_series(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.strip()
    # remove VIDEO_ prefix
    s = s.str.replace(r"^(VIDEO_|video_|Video_)", "", regex=True)
    # keep only digits if there is extra noise (safety)
    s = s.str.extract(r"(\d+)", expand=False).fillna(s)
    return s.str.strip()

# =========================
# 1) Load merged features + y
# =========================
train = read_csv_robust(TRAIN_MERGED)
test  = read_csv_robust(TEST_MERGED)
y_df  = read_csv_robust(Y_PATH)

# Ensure ID column name in y
if "ID" not in y_df.columns:
    # common cases: first col is ID, or 'video_id'
    if "video_id" in y_df.columns:
        y_df = y_df.rename(columns={"video_id": "ID"})
    elif "id" in y_df.columns:
        y_df = y_df.rename(columns={"id": "ID"})
    else:
        y_df = y_df.rename(columns={y_df.columns[0]: "ID"})

# Normalize IDs everywhere
for df in (train, test, y_df):
    if "ID" not in df.columns:
        # try to rename common alternatives
        if "video_id" in df.columns:
            df.rename(columns={"video_id": "ID"}, inplace=True)
        elif "id" in df.columns:
            df.rename(columns={"id": "ID"}, inplace=True)
        else:
            raise ValueError(f"Impossible de trouver la colonne ID dans: {df.columns.tolist()[:30]}")
    df["ID"] = normalize_id_series(df["ID"])

# Drop any existing popularity column in features (avoid leakage/conflict)
if "popularity" in train.columns:
    train = train.drop(columns=["popularity"])
if "popularity" in test.columns:
    test = test.drop(columns=["popularity"])

# Check duplicates in y
if y_df["ID"].duplicated().any():
    dups = y_df.loc[y_df["ID"].duplicated(keep=False), "ID"].value_counts().head(10)
    raise ValueError(f"IDs dupliqués dans y_train (top 10):\n{dups}")

# Merge target
train = train.merge(y_df[["ID", "popularity"]], on="ID", how="left", validate="m:1")

print(train[["ID", "popularity"]].head(5))
print(f"Train shape: {train.shape} | Target nulls: {train['popularity'].isna().sum()}")
print(f"Test shape : {test.shape}")

# If some train IDs didn't find a label => drop them (safe)
if train["popularity"].isna().any():
    missing = train.loc[train["popularity"].isna(), "ID"].nunique()
    print(f"⚠️ {missing} IDs train sans label (drop).")
    train = train.dropna(subset=["popularity"]).reset_index(drop=True)

# =========================
# 2) Align columns train/test (CRUCIAL)
# =========================
# Ensure both have same feature columns (except popularity)
drop_cols = ["ID", "popularity"]

train_cols = set(train.columns) - set(drop_cols)
test_cols  = set(test.columns) - set(["ID"])

# Add missing columns to test/train with NaN
missing_in_test = sorted(list(train_cols - test_cols))
missing_in_train = sorted(list(test_cols - train_cols))

for c in missing_in_test:
    test[c] = np.nan
for c in missing_in_train:
    train[c] = np.nan

# Reorder columns: ID first, then sorted features, then popularity at end (optional)
feature_candidates = sorted(list((set(train.columns) - set(drop_cols))))
train = train[["ID"] + feature_candidates + ["popularity"]]
test  = test[["ID"] + feature_candidates]

# =========================
# 3) Prepare features (numeric only)
# =========================
non_numeric = train[feature_candidates].select_dtypes(exclude=[np.number]).columns.tolist()
print(f"\nColonnes non-numériques droppées ({len(non_numeric)}): {non_numeric[:10]}{'...' if len(non_numeric)>10 else ''}")

feature_cols = [c for c in feature_candidates if c not in non_numeric]

X = train[feature_cols].copy()
y = train["popularity"].astype(float).copy()
X_test_final = test[feature_cols].copy()

print(f"\nNombre de features: {len(feature_cols)}")
print(f"NaN dans X: {int(X.isna().sum().sum())} | NaN dans X_test: {int(X_test_final.isna().sum().sum())}")

# =========================
# 4) LightGBM params + CV
# =========================
lgb_params = {
    "objective":         "regression",
    "metric":            "rmse",
    "learning_rate":     0.03,
    "num_leaves":        150,
    "max_depth":         -1,
    "min_child_samples": 20,
    "feature_fraction":  0.8,
    "bagging_fraction":  0.8,
    "bagging_freq":      5,
    "reg_alpha":         0.1,
    "reg_lambda":        1.0,
    "n_jobs":            -1,
    "verbose":           -1,
    "random_state":      SEED,
}

kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

oof_preds   = np.zeros(len(X))
test_preds  = np.zeros(len(X_test_final))
feature_imp = np.zeros(len(feature_cols))

print(f"\n{'='*50}")
print(f"KFold CV — {N_SPLITS} folds")
print(f"{'='*50}")

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y), 1):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = lgb.LGBMRegressor(n_estimators=3000, **lgb_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=False),
            lgb.log_evaluation(period=200)
        ]
    )

    oof_preds[val_idx] = model.predict(X_val)
    test_preds += model.predict(X_test_final) / N_SPLITS
    feature_imp += model.feature_importances_ / N_SPLITS

    fold_rmse = np.sqrt(mean_squared_error(y_val, oof_preds[val_idx]))
    print(f"  Fold {fold} | Best iter: {model.best_iteration_:4d} | RMSE: {fold_rmse:.4f}")

oof_rmse = np.sqrt(mean_squared_error(y, oof_preds))
print(f"\n{'='*50}")
print(f"OOF RMSE global : {oof_rmse:.4f}")
print(f"{'='*50}")



# =========================
# 6) Submission
# =========================
submission = pd.DataFrame({
    "ID": test["ID"].astype(str).values,
    "popularity": test_preds
})

assert submission["ID"].isna().sum() == 0, "IDs manquants dans la submission!"
assert submission["popularity"].isna().sum() == 0, "Prédictions NaN dans la submission!"

out_path = ROOT / "submission_lgbm.csv"
submission.to_csv(out_path, index=False)

print(f"\n✅ Submission sauvegardée : {out_path}")
print(f"   Shape : {submission.shape}")
print("\nAperçu:")
print(submission.head(10).to_string(index=False))
print("\nStats popularity prédit:")
print(submission["popularity"].describe().round(4))

                    ID  popularity
0  7602656035161050390    8.603777
1  7590718903144287510    8.537734
2  7571821778746592534   11.208495
3  7569927329154190614   11.982352
4  7566270741134462230    8.678487
Train shape: (1348, 1858) | Target nulls: 0
Test shape : (338, 1312)

Colonnes non-numériques droppées (0): []

Nombre de features: 2778
NaN dans X: 1251224 | NaN dans X_test: 496401

KFold CV — 5 folds
[200]	valid_0's rmse: 1.27766
  Fold 1 | Best iter:  128 | RMSE: 1.2696
[200]	valid_0's rmse: 1.23838


KeyboardInterrupt: 